# classpatch — recursive multi-model audio classification

This notebook is organized as:

1. **Imports & paths** — configure audio input, cache, and output paths.
2. **Models** — instantiate classifiers and register them with the pipeline.
3. **Routes** — define downstream routing rules from code or a saved JSON config.
4. **Single-file run** — execute the pipeline on one clip and render the result tree.
5. **Batch + survey** — process all clips, aggregate `(file, model, label)` tallies, and export CSV.

Re-running routing and execution cells does not rebuild models; models are loaded
on first use and reused across later cell executions.

In [ ]:
from pathlib import Path

from classpatch import (
    Pipeline,
    RouteBook,
    SSLAMClassifier,
    StubClassifier,
    Survey,
    render,
)

# Where audio lives, where to cache/persist config and outputs.
AUDIO_DIR = Path("audio/in")
CACHE_DIR = Path("cache")
ROUTES_PATH = CACHE_DIR / "routes.json"
SURVEY_CSV_PATH = CACHE_DIR / "survey.csv"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
assert AUDIO_DIR.is_dir(), f"Audio directory not found: {AUDIO_DIR}"
print(f"Found {len(list(AUDIO_DIR.glob('*.wav')))} .wav files in {AUDIO_DIR}/")

## 2. Models

Register the classifiers used by the pipeline. Model objects are lightweight;
weights are loaded lazily on first inference.

This configuration uses SSLAM (AudioSet, 527 labels) as the entry model and
two `StubClassifier` instances as downstream specialists. Stubs implement the
same `AudioClassifier` interface as production wrappers, so route logic can be
tested end-to-end before integrating specialized models.

To remove a specialist, delete its `register(...)` call and remove any route
rules that target it in later cells.

In [ ]:
# Entry model: SSLAM (AudioSet, 527 labels). max_chunks caps how many of
# the 10-second windows are classified per file; None classifies the whole
# file.
sslam = SSLAMClassifier(max_chunks=None)

# Downstream specialists -- stubs here, real wrappers when available.
stub_birdnet = StubClassifier(
    name="stub_birdnet",
    fixed_predictions=[
        ("(stub) Northern Cardinal", 0.65),
        ("(stub) American Robin", 0.42),
    ],
)
stub_trainnet = StubClassifier(
    name="stub_trainnet",
    fixed_predictions=[
        ("(stub) Diesel locomotive", 0.71),
        ("(stub) Freight train", 0.38),
    ],
)

pipeline = (
    Pipeline(
        max_depth=3,
        report_top_k=8,
        report_score_floor=0.05,
    )
    .register(sslam)
    .register(stub_birdnet)
    .register(stub_trainnet)
)
print("registered models:", list(pipeline.models))

## 3. Routes

Routes define conditional downstream execution, for example: when SSLAM emits
`Bird` above 0.2, run BirdNET on the same window. Label *groups* let a rule
reference ontology subsets (for example, all AudioSet bird labels) without
listing each label repeatedly.

If `cache/routes.json` exists, this cell loads it to preserve prior routing
configuration. Otherwise it creates and saves a default route set. To modify
routes from code again, delete `cache/routes.json`, edit this cell, and rerun.

Example:

```python
book.route("sslam").when("any_bird").above(0.20).to("stub_birdnet")
```

`.when(name)` resolves to a named group when present, otherwise to a literal
label. `.when_group(...)`, `.when_label(...)`, and `.when_any({...})` are
explicit alternatives.

In [ ]:
if ROUTES_PATH.exists():
    book = RouteBook.load(ROUTES_PATH)
    print(f"loaded existing route book from {ROUTES_PATH}")
else:
    book = RouteBook()

    # AudioSet label groups -- edit these in one place to update every rule
    # that references them.
    book.define_group("any_bird", {
        "Bird",
        "Bird vocalization, bird call, bird song",
        "Chirp, tweet",
        "Squawk",
        "Caw",
        "Hoot",
        "Coo",
    })
    book.define_group("any_train", {
        "Train",
        "Rail transport",
        "Train whistle",
        "Train horn",
        "Train wheels squealing",
        "Railroad car, train wagon",
        "Subway, metro, underground",
    })

    book.route("sslam").when("any_bird").above(0.20).to("stub_birdnet")
    book.route("sslam").when("any_train").above(0.20).to("stub_trainnet")

    book.save(ROUTES_PATH)
    print(f"created default route book and saved to {ROUTES_PATH}")

pipeline.routes = book
print(book)
for rule in book.rules:
    print(" -", rule)

## 3b. Label routes (custom ontologies)

A **label route** remaps model outputs into a user-defined ontology. When a
source prediction matches a rule, the pipeline emits a *synthetic* prediction
under the ontology model name. The synthetic prediction keeps the source time
range. If multiple source labels map to the same target label in the same
window, the synthetic score is the **max** contributing score.

This is useful for collapsing broad ontologies (such as AudioSet's 527 labels)
into survey-specific categories:

```python
ecology = book.ontology("ecology")
ecology.relabel("sslam").when("any_bird").above(0.20).as_label("bird_activity")
ecology.relabel("sslam").when_label("Speech").above(0.40).as_label("human_voice")
```

Multiple rules can map into the same target label, and one source label can
map to multiple targets via separate rules. Synthetic predictions appear as
child nodes with `(model = ontology_name)` and as separate survey rows. Use
`models=("ecology",)` in `Survey.summarize` to report only remapped counts.

In [ ]:
if book.label_rules:
    print(f"loaded {len(book.label_rules)} label rule(s) from {ROUTES_PATH}")
else:
    socks = book.ontology("socks")

    # Coarse categories
    socks.relabel("sslam").when("any_bird").as_label("biophony")
    socks.relabel("sslam").when("Speech").as_label("anthrophony")

    # Same source -> multiple target labels (declare one rule per target):
    socks.relabel("sslam").when_label("Train horn").as_label("trains")
    socks.relabel("sslam").when_label("Train horn").as_label("anthrophony")

    book.save(ROUTES_PATH)
    print(f"defined default label routes and re-saved {ROUTES_PATH}")

print(book)
for r in book.label_rules:
    print(" -", r)

## 4. Run on a single file

Pipeline output is a `ResultNode` tree: one node per `(model, segment)`
invocation containing predictions and triggered children. `render(tree)` prints
an indented timestamped view for routing inspection.

This cell runs on the first file in `AUDIO_DIR` and temporarily limits SSLAM
to the first 3 windows (about 30 seconds). Set `sslam.max_chunks = None` to
process the full file.

In [ ]:
files = sorted(AUDIO_DIR.glob("*.wav"))
assert files, f"No .wav files in {AUDIO_DIR}/"

# Limit SSLAM windows in this cell for faster turnaround.
sslam.max_chunks = 3

target = files[0]
print(f"running on {target.name} ...")
tree = pipeline.run(target, entry_model="sslam")

print()
print(render(tree, top_k=5))

## 5. Batch over the folder + survey

`Pipeline.run_many(paths, entry_model=...)` returns a `{Path: ResultNode}`
mapping compatible with `Survey.summarize(...)`. The survey aggregates
predictions into `(file, model, label)` rows with detection/event counts,
union duration, and score statistics.

Defaults below:

- `min_score=0.20` — exclude low-confidence predictions before counting.
- `merge_gap_sec=None` — report raw **detection count** (set a float such as
  `2.0` to merge adjacent same-label detections into events).
- `models=None` — include labels from all models. Use `models=("sslam",)` for
  SSLAM-only counts, or `models=("stub_birdnet",)` for species-only counts,
  to avoid counting a "Bird -> Cardinal" event at multiple ontology levels.

Tallies are written to `cache/survey.csv`. Filenames are preserved as-is so
location metadata can be derived from filename conventions downstream.

In [ ]:
# Maximum number of chunks (for sslam one chunk is 10 seconds) to process for each file
# None processes the whole file
sslam.max_chunks = None

trees = pipeline.run_many(files, entry_model="sslam")
print(f"processed {len(trees)} file(s)")

survey = Survey.summarize(
    trees,
    min_score=0.20,
    merge_gap_sec=0.0,   # set e.g. 2.0 to also collapse adjacent detections, None disables merge
)
print(survey)
print()
print("Top 12 (file, model, label) by detection count:")
for t in survey.top(12):
    print(
        f"  {t.source_path.name:<60s} {t.model:<14s} "
        f"{t.label:<32s} detections={t.detection_count:<3d} "
        f"dur={t.total_duration_sec:>6.1f}s  max={t.max_score:.2f}"
    )

written = survey.to_csv(SURVEY_CSV_PATH)
print(f"\nwrote {len(survey.tallies)} tallies to {written}")